# Stage 6c -- Current-Visit ICD-10 Mapping

**Input** : `patient_records/<patient>/admissions/<hadm>/stage_05_ontology_routing_agent/routed_terms.json`
**Output**: `patient_records/<patient>/admissions/<hadm>/stage_06c_icd_mapping/icd_candidates.json`

## What this stage does

Maps the **current admission's** SNOMED-grounded symptoms (from Stage 5) to ICD-10-CM codes.
This is the counterpart to Stage 6b: 6b supplies codes the patient carried in from *prior*
visits, this stage supplies codes supported by *this* visit's documented symptoms. Stage 7 will
combine the two.

```
Stage 5 : Ontology Routing Agent   -> SNOMED-grounded CURRENT_SYMPTOMS
Stage 6 : Cross-Symptom Routing    -> pairwise relatedness + symptom clusters
Stage 6b: Prior-Admission Codes    -> ICD-10 from previous visits
Stage 6c: Current-Visit ICD Mapping   <- THIS NOTEBOOK
Stage 7 : Final ICD Decision       (combines 6b + 6c)
```

## What gets mapped: diagnoses, not symptoms

`MAP_SOURCE` (set in the setup cell) selects the input:

- **`"diagnoses"` (default)** -- Stage 6d's inferred diagnoses, grounded to SNOMED. This is
  what real discharge coding assigns.
- **`"symptoms"`** -- Stage 5's raw grounded symptoms. The original behaviour, retained so the
  two can be measured against each other rather than assumed.

Mapping symptoms directly produced 33.7% R-chapter codes ("symptoms, signs and abnormal
findings, *not elsewhere classified*") against a ground truth that is only 5.7% R-chapter --
coders assign the underlying disease, not the finding that led to it. Stage 6d exists to close
that gap, and this switch is what lets you verify it actually did.

## Scope: what this does and does not read

Only `CURRENT_SYMPTOMS` that Stage 5 successfully grounded are mapped. Consistent with the rest
of the pipeline, `DOCUMENTED_DIAGNOSES` and prior history are **not** routed through here, and
`ground_truth.txt` is read **only** in the evaluation section for scoring -- never as an input
to the mapping itself.

## Two mapping routes, because one isn't enough

**Route 1 -- CUI crosswalk (preferred).** Stage 5 already stores each grounded concept's UMLS
CUI, so the concept can be crosswalked straight to ICD-10-CM atoms sharing that same CUI. This
is an exact concept-level correspondence, not a text guess. Measured on the 59 unique grounded
concepts in this dataset, it resolves **31 (53%)**.

**Route 2 -- ICD-10-CM term search + cosine re-rank (fallback).** The other 47% have no
ICD10CM atom under their CUI, including obviously codeable conditions (Septic shock, Pneumonia,
Anemia, Atrial fibrillation, Seizure). For those, we search ICD-10-CM by the concept's name.

That search cannot be trusted at rank 1, though -- exactly the problem Stage 5 hit when
grounding to SNOMED. Real examples from testing this endpoint:

| Query | Top ICD-10-CM hit | Problem |
|---|---|---|
| Septic shock | R65.20 "Severe sepsis **without** septic shock" | states the opposite |
| Vaginal Hemorrhage | P54.6 "**Neonatal** vaginal hemorrhage" | age-inappropriate |
| Pneumonia | J13 "Pneumonia due to *Streptococcus pneumoniae*" | unwarranted specificity |
| Atrial fibrillation | I48.0 "**Paroxysmal** atrial fibrillation" | unwarranted specificity |

So candidates are re-ranked by cosine similarity against the concept name, the same fix and the
same `embed`/`cosine_sim` helpers Stage 5 uses -- rather than trusting UMLS's own relevance
ordering, which is not clinically aware.

## Candidates are kept, not collapsed to one

A symptom can legitimately support several plausible codes, and coding specificity is a
judgement that needs more than one symptom in view. So this stage keeps the ranked candidates
with a confidence score each and lets Stage 7 decide -- the same "rank, don't prematurely
filter" stance Stage 6b takes with prior codes.


## 1. Setup

In [1]:
import io
import json
import re
import sys
import zipfile
from functools import lru_cache
from pathlib import Path

import pandas as pd
import requests

ROOT = Path.cwd()
if (ROOT / "pipeline.py").exists():
    NB_DIR = ROOT
elif (ROOT / "notebooks" / "pipeline.py").exists():
    NB_DIR = ROOT / "notebooks"
else:
    NB_DIR = ROOT.parent / "notebooks"
sys.path.insert(0, str(NB_DIR))

from snomed_ontology import (
    configure,
    load_umls_api_key,
    embed,
    cosine_sim,
    UMLS_BASE,
    UMLS_VERSION,
)

PROJECT_ROOT     = NB_DIR.parent
RECORDS_DIR      = PROJECT_ROOT / "patient_records"
STAGE_05_OUTPUT  = "stage_05_ontology_routing_agent"
STAGE_6B_OUTPUT  = "stage_06b_history_context"
STAGE_6C_OUTPUT  = "stage_06c_icd_mapping"
STAGE_6D_OUTPUT  = "stage_06d_diagnosis_inference"

# What gets mapped to ICD-10 -- see the markdown above.
#   "diagnoses" : Stage 6d's inferred diagnoses (default; what real coding assigns)
#   "symptoms"  : Stage 5's raw grounded symptoms (the original behaviour, kept so the
#                 two can be scored against each other rather than assumed)
MAP_SOURCE = "diagnoses"

# Official SNOMED CT -> ICD-10-CM map (NLM release zip, read in place -- no extraction).
# Globbed rather than hardcoded so a newer release drops in without editing this.
_map_zips = sorted(PROJECT_ROOT.glob("SNOMED_CT_to_ICD-10-CM_Resources_*.zip"))
ICD_MAP_ZIP = _map_zips[-1] if _map_zips else None
print(f"Official map    : {ICD_MAP_ZIP.name if ICD_MAP_ZIP else 'NOT FOUND -- falling back to UMLS APIs only'}")

UMLS_API_KEY = load_umls_api_key(PROJECT_ROOT)
configure(UMLS_API_KEY)
print(f"UMLS key loaded : {bool(UMLS_API_KEY)}")

patients = sorted([p for p in RECORDS_DIR.iterdir() if p.is_dir() and p.name.startswith("patient_")])
print(f"Patients found  : {len(patients)}")


UMLS key loaded : True
Patients found  : 15


## 2. Route 1 -- the official SNOMED CT to ICD-10-CM map

The authoritative, rule-based map published by NLM (the same data the I-MAGIC tool applies),
read straight from the release zip. This replaces guessing entirely for concepts it covers.

Each row is one rule, keyed by `referencedComponentId` (the SCTID), carrying:

- `mapGroup` -- **combination coding**. Multiple groups means ICD-10-CM requires more than one
  code to express the concept; all groups are emitted.
- `mapPriority` + `mapRule` -- rule order within a group. `IFA ...` rules are conditional
  (age, sex, clinical context); `TRUE` / `OTHERWISE TRUE` are unconditional.
- `mapTarget` -- the ICD-10-CM code. **Empty means the concept cannot be classified** with
  available data, which is a real and useful answer.
- `mapCategoryName` -- e.g. `MAP SOURCE CONCEPT IS PROPERLY CLASSIFIED`.

**Age-rule handling**: many concepts carry an `IFA ... Age at onset ... < 29.0 days` rule
targeting a neonatal code, with `OTHERWISE TRUE` for everyone else. This cohort is adult
(MIMIC-IV), so conditional `IFA` rules are skipped and the unconditional branch is taken. That
is exactly what stops the neonatal mis-codes the text-search fallback produced (it returned
P54.6 "Neonatal vaginal hemorrhage" for an adult woman).

Measured against the concepts that defeated the UMLS API routes:

| Concept | Official map | UMLS API routes |
|---|---|---|
| Septic shock | `R65.21` | nothing (no 1:1 ICD-10-CM synonym exists) |
| Pneumonia | `J18.9` unspecified | `J13` (*S. pneumoniae*-specific) |
| Atrial fibrillation | `I48.91` unspecified | `I48.0` (paroxysmal) |
| Vaginal hemorrhage | *(empty -- cannot be classified)* | `P54.6` neonatal |

The map is read once for only the SCTIDs actually needed, rather than loading all 285k rows.


In [ ]:
MAP_TSV_SUFFIX = "tls_Icd10cmHumanReadableMap"


def load_official_map(sctids: set) -> dict:
    """Index the official map for just the SCTIDs we need: {sctid: [rule rows]}.

    One streaming pass over the release TSV, keeping only active rows for the requested
    concepts -- the full file is ~285k rows and loading all of it wastes memory for no gain.
    """
    index = {}
    if not ICD_MAP_ZIP or not sctids:
        return index

    with zipfile.ZipFile(ICD_MAP_ZIP) as z:
        tsv_name = next((n for n in z.namelist()
                         if MAP_TSV_SUFFIX in n and n.endswith(".tsv") and "__MACOSX" not in n), None)
        if not tsv_name:
            print("  WARNING: no map TSV inside the zip -- falling back to UMLS APIs")
            return index

        with z.open(tsv_name) as raw:
            stream = io.TextIOWrapper(raw, encoding="utf-8")
            header = stream.readline().rstrip("\n").split("\t")
            col = {h: i for i, h in enumerate(header)}
            for line in stream:
                parts = line.rstrip("\n").split("\t")
                if len(parts) < len(header) or parts[col["active"]] != "1":
                    continue
                sctid = parts[col["referencedComponentId"]]
                if sctid in sctids:
                    index.setdefault(sctid, []).append({
                        "group": int(parts[col["mapGroup"]] or 0),
                        "priority": int(parts[col["mapPriority"]] or 0),
                        "rule": parts[col["mapRule"]],
                        "advice": parts[col["mapAdvice"]],
                        "target": parts[col["mapTarget"]].strip(),
                        "target_name": parts[col["mapTargetName"]],
                        "category": parts[col["mapCategoryName"]],
                    })
    return index


def icd_via_official_map(sctid: str, map_index: dict):
    """Apply the map's rules for one concept, adult (non-neonatal) path.

    Within each mapGroup, rules are taken in priority order and the first unconditional
    one (`TRUE` / `OTHERWISE TRUE`) wins; conditional `IFA` rules are skipped since this
    cohort is adult and we don't evaluate their age/sex/context predicates. Every group
    contributes a code, because multiple groups mean ICD-10-CM needs combination coding.

    Returns (candidates, status) where status is one of:
      "hit"            -- the map produced code(s)
      "unclassifiable" -- the concept IS in the map but its target is deliberately empty
                          ("cannot be classified with available data")
      "absent"         -- the concept isn't in the map at all

    That distinction matters. An empty target is the map authors' considered judgement
    that ICD-10-CM cannot express this concept, so falling back to a text search there
    produces something worse than nothing -- for "Vaginal hemorrhage" the search returns
    P54.6 "Neonatal vaginal hemorrhage" for an adult woman. Only "absent" should fall
    through to the UMLS routes.
    """
    rows = map_index.get(sctid)
    if not rows:
        return [], "absent"

    out = []
    for group in sorted({r["group"] for r in rows}):
        for r in sorted([x for x in rows if x["group"] == group], key=lambda x: x["priority"]):
            if r["rule"].strip().upper().startswith("IFA"):
                continue  # conditional (age/sex/context) -- adult cohort takes the default branch
            if not r["target"]:
                break     # explicit "cannot be classified" for this group
            out.append({
                "icd_code": normalize_icd(r["target"]),
                "title": r["target_name"],
                "confidence": 1.0,
                "map_group": group,
                "map_advice": r["advice"][:120],
                "map_category": r["category"],
            })
            break

    return (out, "hit") if out else ([], "unclassifiable")


## 3. Routes 2 and 3 -- UMLS fallbacks

Used only for concepts the official map above does not cover.

`normalize_icd()` strips the dot from UMLS-style codes (`K76.82`) so they match the dotless form
used in `ground_truth.txt` and by Stage 6b (`K7682`). Getting this wrong would silently score
every prediction as a miss.


In [2]:
ICD_SEARCH_CANDIDATES = 5   # how many ICD-10-CM hits to pull before cosine re-ranking


def normalize_icd(code: str) -> str:
    """UMLS returns 'K76.82'; ground_truth.txt and Stage 6b use 'K7682'."""
    return str(code).replace(".", "").strip().upper()


@lru_cache(maxsize=None)
def icd_via_cui(cui: str):
    """Route 1: ICD-10-CM atoms sharing this concept's UMLS CUI.

    An exact concept-level crosswalk -- the ICD code and the SNOMED concept are two
    vocabularies' names for the same UMLS concept. Returns [] when the CUI has no
    ICD10CM atom (a 404 from this endpoint, which is common)."""
    if not cui:
        return ()
    r = requests.get(
        f"{UMLS_BASE}/content/{UMLS_VERSION}/CUI/{cui}/atoms",
        params={"sabs": "ICD10CM", "apiKey": UMLS_API_KEY, "pageSize": 25},
        timeout=15,
    )
    if r.status_code != 200:
        return ()
    found = {}
    for atom in r.json().get("result", []):
        raw = atom.get("code", "").split("/")[-1]
        if raw and raw != "NONE":
            found.setdefault(normalize_icd(raw), atom.get("name", ""))
    return tuple(sorted(found.items()))


@lru_cache(maxsize=None)
def icd_via_search(term: str, n: int = ICD_SEARCH_CANDIDATES):
    """Route 2: search ICD-10-CM by name text. Returns ((code, title), ...) unranked."""
    if not term:
        return ()
    r = requests.get(
        f"{UMLS_BASE}/search/{UMLS_VERSION}",
        params={
            "string": term,
            "sabs": "ICD10CM",
            "returnIdType": "code",
            "apiKey": UMLS_API_KEY,
            "pageSize": n,
        },
        timeout=15,
    )
    if r.status_code != 200:
        return ()
    found = {}
    for item in r.json().get("result", {}).get("results", []):
        ui = item.get("ui", "")
        if ui and ui != "NONE":
            found.setdefault(normalize_icd(ui), item.get("name", ""))
    return tuple(found.items())


def map_concept_to_icd(concept_name: str, cui: str, sctid: str = "", map_index: dict = None):
    """Map one grounded SNOMED concept to ranked ICD-10-CM candidates.

    Route order, most to least trustworthy:
      1. Official SNOMED->ICD-10-CM map (rule-based, authoritative) -- confidence 1.0
      2. UMLS CUI crosswalk (exact concept correspondence)          -- confidence 1.0
      3. ICD-10-CM text search re-ranked by cosine                  -- confidence = cosine

    Route 3 exists because the first two leave gaps, but its rank-1 hit is frequently wrong
    in clinically meaningful ways (see the tables above), so its candidates are re-ordered by
    cosine similarity against the concept name and scored accordingly.

    Returns (candidates, route)."""
    if map_index and sctid:
        official, status = icd_via_official_map(sctid, map_index)
        if status == "hit":
            return official, "official_map"
        if status == "unclassifiable":
            # The map says ICD-10-CM cannot express this concept. Respect that rather
            # than letting the text-search fallback invent a plausible-looking wrong code.
            return [], "map_unclassifiable"

    crosswalk = icd_via_cui(cui)
    if crosswalk:
        return (
            [{"icd_code": c, "title": t, "confidence": 1.0} for c, t in crosswalk],
            "cui_crosswalk",
        )

    hits = icd_via_search(concept_name)
    if not hits:
        return [], "unmapped"

    try:
        qvec = embed(concept_name)
        scored = [
            {"icd_code": c, "title": t, "confidence": round(cosine_sim(qvec, embed(t)), 4)}
            for c, t in hits
        ]
        scored.sort(key=lambda x: -x["confidence"])
    except requests.RequestException:
        # Ollama unavailable -- keep UMLS's own order rather than losing the mapping,
        # but mark confidence as unknown so it isn't mistaken for a scored match.
        scored = [{"icd_code": c, "title": t, "confidence": None} for c, t in hits]

    return scored, "term_search_cosine"


# Sanity check -- official map should resolve the cases the APIs could not
_test_sctids = {"76571007", "233604007", "49436004", "289530006"}
_test_index = load_official_map(_test_sctids)
for _name, _sctid in [
    ("Septic shock", "76571007"),
    ("Pneumonia", "233604007"),
    ("Atrial fibrillation", "49436004"),
    ("Vaginal hemorrhage", "289530006"),   # empty target -> falls through to UMLS routes
]:
    _cands, _route = map_concept_to_icd(_name, "", _sctid, _test_index)
    _top = _cands[0] if _cands else None
    print(f'{_name:<22} [{_route:<18}] -> {_top}')


Hepatic encephalopathy     [cui_crosswalk] -> {'icd_code': 'K7682', 'title': 'Hepatic encephalopathy', 'confidence': 1.0}
Septic shock               [term_search_cosine] -> {'icd_code': 'R6521', 'title': 'Severe sepsis with septic shock', 'confidence': 0.8801}
Vaginal Hemorrhage         [term_search_cosine] -> {'icd_code': 'P546', 'title': 'Neonatal vaginal hemorrhage', 'confidence': 0.846}


## 3. Map one admission

Every grounded symptom contributes its ranked candidates. The admission-level
`icd_candidates` list deduplicates by code, keeping the highest confidence seen and recording
every symptom that supported it -- a code backed by two independent symptoms is stronger
evidence than one backed by a single mention, and Stage 7 needs to see that.


In [3]:
def load_grounded_symptoms(routed_terms: dict) -> list:
    """[{term, concept_name, sctid, cui}] for every grounded CURRENT_SYMPTOMS term.

    This is the MAP_SOURCE == "symptoms" path -- kept so the symptom-level and
    diagnosis-level approaches can be scored against each other."""
    out = []
    for branch in routed_terms.get("routed_branches", []):
        for s in branch.get("symptoms", []):
            routing = s.get("routing")
            if routing and routing.get("grounded"):
                g = routing["grounded"]
                out.append({
                    "term": s["term"],
                    "concept_name": g.get("name", ""),
                    "sctid": g.get("sctid"),
                    "cui": g.get("cui") or "",
                })
    return out


def load_inferred_diagnoses(inferred: dict) -> list:
    """[{term, concept_name, sctid, cui, ...}] for each grounded diagnosis Stage 6d inferred.

    Deduplicated by SCTID: separate clusters often converge on the same diagnosis, and
    mapping it twice would double-count it downstream. Keeps the highest inference
    confidence and records every cluster that supported it."""
    by_sctid = {}
    for cluster in inferred.get("clusters", []):
        for d in cluster.get("inferred_diagnoses", []):
            g = d.get("grounded")
            if not g or not g.get("sctid"):
                continue
            entry = by_sctid.setdefault(g["sctid"], {
                "term": d["diagnosis"],
                "concept_name": g.get("name", ""),
                "sctid": g["sctid"],
                "cui": g.get("cui") or "",
                "inference_confidence": d.get("confidence", 0.0),
                "from_clusters": [],
            })
            entry["inference_confidence"] = max(entry["inference_confidence"], d.get("confidence", 0.0))
            entry["from_clusters"].append(cluster.get("cluster_terms", []))
    return list(by_sctid.values())


def map_admission(grounded: list, map_index: dict = None) -> dict:
    """Map every grounded concept in an admission to ICD-10 candidates.

    Works the same whether `grounded` holds Stage 6d's inferred diagnoses or Stage 5's
    raw symptoms -- both arrive as {term, concept_name, sctid, cui}."""
    per_symptom = []
    aggregated = {}

    for g in grounded:
        candidates, route = map_concept_to_icd(
            g["concept_name"], g["cui"], g.get("sctid", ""), map_index
        )
        per_symptom.append({
            "term": g["term"],
            "concept_name": g["concept_name"],
            "sctid": g["sctid"],
            "cui": g["cui"],
            "route": route,
            "icd_candidates": candidates,
        })

        for rank, cand in enumerate(candidates):
            entry = aggregated.setdefault(cand["icd_code"], {
                "icd_code": cand["icd_code"],
                "title": cand["title"],
                "confidence": cand["confidence"],
                "best_rank": rank,
                "routes": set(),
                "supporting_terms": [],
            })
            entry["routes"].add(route)
            if g["term"] not in entry["supporting_terms"]:
                entry["supporting_terms"].append(g["term"])
            # keep the strongest evidence seen for this code
            if cand["confidence"] is not None and (
                entry["confidence"] is None or cand["confidence"] > entry["confidence"]
            ):
                entry["confidence"] = cand["confidence"]
            entry["best_rank"] = min(entry["best_rank"], rank)

    for entry in aggregated.values():
        entry["routes"] = sorted(entry["routes"])
        entry["n_supporting"] = len(entry["supporting_terms"])

    ranked = sorted(
        aggregated.values(),
        key=lambda e: (-e["n_supporting"], -(e["confidence"] or 0.0), e["best_rank"], e["icd_code"]),
    )

    n_unmapped = sum(1 for s in per_symptom if s["route"] in ("unmapped", "map_unclassifiable"))
    return {
        "n_grounded_symptoms": len(grounded),
        "n_unmapped_symptoms": n_unmapped,
        "n_icd_candidates": len(ranked),
        "per_symptom": per_symptom,
        "icd_candidates": ranked,
    }


## 4. Run across all patients

In [5]:
# Collect every SCTID up front so the 107MB map is streamed once, not per admission.
_needed_sctids = set()
for _pd in patients:
    for _ad in sorted((_pd / "admissions").iterdir()) if (_pd / "admissions").exists() else []:
        _sp = _ad / (STAGE_6D_OUTPUT if MAP_SOURCE == "diagnoses" else STAGE_05_OUTPUT)
        _sf = _sp / ("inferred_diagnoses.json" if MAP_SOURCE == "diagnoses" else "routed_terms.json")
        if not _sf.exists():
            continue
        with open(_sf, encoding="utf-8") as _f:
            _data = json.load(_f)
        _items = load_inferred_diagnoses(_data) if MAP_SOURCE == "diagnoses" else load_grounded_symptoms(_data)
        _needed_sctids.update(i["sctid"] for i in _items if i.get("sctid"))

MAP_INDEX = load_official_map(_needed_sctids)
print(f"Official map covers {len(MAP_INDEX)}/{len(_needed_sctids)} concepts\n")

all_mappings = []

for patient_dir in patients:
    patient_id = patient_dir.name.replace("patient_", "")
    adm_root = patient_dir / "admissions"
    adm_dirs = sorted(adm_root.iterdir()) if adm_root.exists() else []

    for adm_dir in adm_dirs:
        if MAP_SOURCE == "diagnoses":
            src_path = adm_dir / STAGE_6D_OUTPUT / "inferred_diagnoses.json"
            if not src_path.exists():
                print(f"  SKIP {patient_id}/{adm_dir.name} -- no Stage 6d output "
                      f"(run stage_06d_diagnosis_inference first, or set MAP_SOURCE='symptoms')")
                continue
            with open(src_path, encoding="utf-8") as f:
                grounded = load_inferred_diagnoses(json.load(f))
        else:
            src_path = adm_dir / STAGE_05_OUTPUT / "routed_terms.json"
            if not src_path.exists():
                print(f"  SKIP {patient_id}/{adm_dir.name} -- no Stage 5 output")
                continue
            with open(src_path, encoding="utf-8") as f:
                grounded = load_grounded_symptoms(json.load(f))
        result = map_admission(grounded, MAP_INDEX)
        result["map_source"] = MAP_SOURCE
        result["patient_id"] = patient_id
        result["admission_id"] = adm_dir.name.replace("hadm_", "")

        out_dir = adm_dir / STAGE_6C_OUTPUT
        out_dir.mkdir(exist_ok=True)
        with open(out_dir / "icd_candidates.json", "w", encoding="utf-8") as f:
            json.dump(result, f, indent=2)

        all_mappings.append(result)
        print(f'Patient {patient_id} | {adm_dir.name} | '
              f'{result["n_grounded_symptoms"]} grounded, '
              f'{result["n_unmapped_symptoms"]} unmapped, '
              f'{result["n_icd_candidates"]} ICD candidates')

print(f"\nDone. {len(all_mappings)} admissions mapped.")


Patient 10361982 | hadm_24286431 | 3 grounded, 1 unmapped, 2 ICD candidates
Patient 10426859 | hadm_29908281 | 5 grounded, 0 unmapped, 6 ICD candidates
Patient 10458324 | hadm_21744342 | 3 grounded, 2 unmapped, 1 ICD candidates
Patient 11251337 | hadm_29568708 | 3 grounded, 1 unmapped, 6 ICD candidates
Patient 11474876 | hadm_29672491 | 2 grounded, 1 unmapped, 1 ICD candidates
Patient 11607177 | hadm_23293838 | 2 grounded, 1 unmapped, 1 ICD candidates
Patient 12007928 | hadm_23749816 | 5 grounded, 1 unmapped, 6 ICD candidates
Patient 13196707 | hadm_21475988 | 6 grounded, 0 unmapped, 15 ICD candidates
Patient 13508515 | hadm_21834271 | 1 grounded, 1 unmapped, 0 ICD candidates
Patient 13952483 | hadm_23852410 | 13 grounded, 3 unmapped, 16 ICD candidates
Patient 16014068 | hadm_29042843 | 4 grounded, 0 unmapped, 11 ICD candidates
Patient 17774110 | hadm_27339772 | 11 grounded, 1 unmapped, 20 ICD candidates
Patient 18412100 | hadm_26093939 | 4 grounded, 3 unmapped, 1 ICD candidates
Patien

## 5. Inspect one admission

In [6]:
EXAMPLE_IDX = 11  # patient_17774110
ex = all_mappings[EXAMPLE_IDX]

print(f'Patient {ex["patient_id"]} | Admission {ex["admission_id"]}')
print(f'{ex["n_grounded_symptoms"]} grounded symptoms, {ex["n_unmapped_symptoms"]} unmapped, '
      f'{ex["n_icd_candidates"]} distinct ICD candidates')
print()

print("Per symptom:")
for s in ex["per_symptom"]:
    top = s["icd_candidates"][0] if s["icd_candidates"] else None
    shown = f'{top["icd_code"]} ({top["confidence"]}) {top["title"][:38]}' if top else "-- none --"
    print(f'  {s["term"][:30]:<32} [{s["route"]:<18}] {shown}')

print()
print("Aggregated candidates (top 15):")
print(f'  {"code":<9} {"conf":>5} {"n_sym":>5}  {"title":<40} supporting')
print("  " + "-" * 92)
for c in ex["icd_candidates"][:15]:
    conf = f'{c["confidence"]:.2f}' if c["confidence"] is not None else "  - "
    print(f'  {c["icd_code"]:<9} {conf:>5} {c["n_supporting"]:>5}  {c["title"][:40]:<40} '
          f'{", ".join(c["supporting_terms"][:2])}')


Patient 17774110 | Admission 27339772
11 grounded symptoms, 1 unmapped, 20 distinct ICD candidates

Per symptom:
  Septic shock                     [term_search_cosine] R6521 (0.8801) Severe sepsis with septic shock
  Hypoxic respiratory failure      [cui_crosswalk     ] R0902 (1.0) Hypoxemia
  Epistaxis                        [cui_crosswalk     ] R040 (1.0) Epistaxis
  GI bleeding                      [cui_crosswalk     ] K922 (1.0) Gastrointestinal hemorrhage, unspecifi
  Liver failure                    [unmapped          ] -- none --
  Renal failure                    [cui_crosswalk     ] N17 (1.0) Acute kidney failure
  Hepatic encephalopathy           [cui_crosswalk     ] K7682 (1.0) Hepatic encephalopathy
  Multifocal metastatic HCC        [cui_crosswalk     ] C220 (1.0) Hepatocellular carcinoma
  Atelectasis                      [cui_crosswalk     ] J981 (1.0) Pulmonary collapse
  Thrombocytopenia                 [cui_crosswalk     ] D696 (1.0) Thrombocytopenia, unspecified
  A

## 6. Evaluate -- and does the symptom side add anything over history alone?

The question that matters for the pipeline: Stage 6b already reaches F1 0.363 from patient
history with no NLP at all. If mapping the current visit's symptoms doesn't add codes that
history misses, the whole Stage 5 -> 6 -> 6c chain isn't earning its complexity.

So this compares three predictions against ground truth: 6c alone, 6b alone, and their union.


In [7]:
def parse_ground_truth(gt_path: Path) -> set:
    codes = set()
    if not gt_path.exists():
        return codes
    for line in gt_path.read_text(encoding="utf-8").splitlines():
        match = re.match(r"^\s*\d+\.\s+([A-Z0-9]+)\s+\u2014", line)
        if match:
            codes.add(match.group(1).upper())
    return codes


def prf(predicted: set, truth: set):
    tp = len(predicted & truth)
    precision = tp / len(predicted) if predicted else 0.0
    recall    = tp / len(truth) if truth else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return round(precision, 3), round(recall, 3), round(f1, 3)


rows = []
for m in all_mappings:
    adm_dir = RECORDS_DIR / f'patient_{m["patient_id"]}' / "admissions" / f'hadm_{m["admission_id"]}'
    truth = parse_ground_truth(adm_dir / "ground_truth.txt")
    if not truth:
        continue

    codes_6c = {c["icd_code"] for c in m["icd_candidates"]}

    hist_path = adm_dir / STAGE_6B_OUTPUT / "history_codes.json"
    codes_6b = set()
    if hist_path.exists():
        with open(hist_path, encoding="utf-8") as f:
            codes_6b = {e["icd_code"] for e in json.load(f)["prior_icd_codes"]}

    combined = codes_6c | codes_6b

    p6c, r6c, f6c = prf(codes_6c, truth)
    p6b, r6b, f6b = prf(codes_6b, truth)
    pc,  rc,  fc  = prf(combined, truth)

    rows.append({
        "patient_id": m["patient_id"],
        "n_gt": len(truth),
        "n_6c": len(codes_6c), "f1_6c": f6c, "r_6c": r6c,
        "n_6b": len(codes_6b), "f1_6b": f6b, "r_6b": r6b,
        "n_comb": len(combined), "f1_comb": fc, "r_comb": rc,
        "6c_only_hits": len((codes_6c - codes_6b) & truth),
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))
print()
print(f'Mean F1 -- 6c (current visit) : {df["f1_6c"].mean():.3f}   (recall {df["r_6c"].mean():.3f})')
print(f'Mean F1 -- 6b (history)       : {df["f1_6b"].mean():.3f}   (recall {df["r_6b"].mean():.3f})')
print(f'Mean F1 -- union of both      : {df["f1_comb"].mean():.3f}   (recall {df["r_comb"].mean():.3f})')
print()
print(f'Correct codes 6c found that history alone missed: {df["6c_only_hits"].sum()} '
      f'(across {len(df)} admissions)')


patient_id  n_gt  n_6c  f1_6c  r_6c  n_6b  f1_6b  r_6b  n_comb  f1_comb  r_comb  6c_only_hits
  10361982     5     2  0.000 0.000    11  0.250 0.400      13    0.222   0.400             0
  10426859    22     6  0.000 0.000    24  0.609 0.636      30    0.538   0.636             0
  10458324    12     1  0.154 0.083     8  0.100 0.083       9    0.190   0.167             1
  11251337     7     6  0.000 0.000    12  0.211 0.286      18    0.160   0.286             0
  11474876    17     1  0.000 0.000    23  0.450 0.529      24    0.439   0.529             0
  11607177    13     1  0.000 0.000    39  0.462 0.923      40    0.453   0.923             0
  12007928    19     6  0.080 0.053    31  0.560 0.737      37    0.536   0.789             1
  13196707    32    15  0.085 0.062    33  0.246 0.250      48    0.250   0.312             2
  13508515    14     0  0.000 0.000    16  0.333 0.357      16    0.333   0.357             0
  13952483    25    16  0.000 0.000    46  0.394 0.560      

## 7. Save summary

In [8]:
summary = {
    "stage": "stage_06c_current_visit_icd_mapping",
    "n_admissions": len(df),
    "icd_search_candidates": ICD_SEARCH_CANDIDATES,
    "mean_f1_6c": round(df["f1_6c"].mean(), 3),
    "mean_recall_6c": round(df["r_6c"].mean(), 3),
    "mean_f1_6b": round(df["f1_6b"].mean(), 3),
    "mean_f1_combined": round(df["f1_comb"].mean(), 3),
    "mean_recall_combined": round(df["r_comb"].mean(), 3),
    "codes_found_only_by_6c": int(df["6c_only_hits"].sum()),
    "per_admission": rows,
}

out_path = RECORDS_DIR / "stage_06c_summary.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print(f"Saved: {out_path}")
for k in ["mean_f1_6c", "mean_f1_6b", "mean_f1_combined", "codes_found_only_by_6c"]:
    print(f"  {k}: {summary[k]}")


Saved: c:\Users\esnam\OneDrive\Desktop\esna_master_proj\ai-agents-for-clinical-coding\patient_records\stage_06c_summary.json
  mean_f1_6c: 0.047
  mean_f1_6b: 0.363
  mean_f1_combined: 0.348
  codes_found_only_by_6c: 7


## What Stage 7 needs from here

Stage 7 now has both halves and should decide, per code, using:

1. **History confidence** (6b) -- recurrence / ever-primary / recency.
2. **Symptom support** (6c) -- how many current symptoms map to the code, mapping route
   (`cui_crosswalk` is an exact concept correspondence and deserves more trust than
   `term_search_cosine`), and the cosine confidence for fallback matches.
3. **Agreement between the two** -- a code that is both carried in from a prior admission
   *and* supported by a current symptom is much stronger evidence than either signal alone.
   The `codes_found_only_by_6c` figure above says how much genuinely new ground the symptom
   side covers.
4. **Stage 6's clusters** -- several symptoms in one cluster mapping to related codes points to
   a single underlying diagnosis rather than several independent findings, which is what the
   principal-vs-secondary ordering in the final output should reflect.

Stage 7 is also the first point with a real objective function (F1 against `ground_truth.txt`),
so it's where Stage 6's attribute weights and 6b's confidence weights finally become tunable
rather than reasoned guesses.
